# 03 - RAGAS Deep Dive : Analyse critique (AWS Bedrock)

Ce notebook explore les limites de RAGAS :
1. Sensibilité au LLM-juge (Claude Sonnet vs Claude Haiku comme évaluateur)
2. Comparaison RAGAS vs évaluation humaine
3. Identification des cas d'échec

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import EXPERIMENT_CONFIGS, AWS_REGION
from src.rag_pipeline import RAGPipeline
from src.evaluation import compare_judges, get_ragas_llm, get_ragas_embeddings
from src.data_loader import load_corpus_from_texts, load_testset

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

## 1. Sensibilité au LLM-juge

RAGAS utilise un LLM pour évaluer les réponses. **Le choix du LLM-juge influence-t-il les scores ?**

On compare :
- Claude Sonnet 4 (plus capable, plus cher)
- Claude Haiku 4.5 (plus rapide, moins cher)
- Claude Haiku 3 (encore plus léger)

In [ ]:
documents = load_corpus_from_texts('../data/corpus')
questions, ground_truths = load_testset('../data/testset/testset.json')

# Pipeline baseline
config = EXPERIMENT_CONFIGS[1]
pipeline = RAGPipeline(config)
pipeline.load_documents(documents)
pipeline.chunk_documents()
pipeline.build_index()

# Sous-ensemble pour l'analyse (10 questions)
subset_q = questions[:10]
subset_gt = ground_truths[:10]

print(f"Testing judge sensitivity on {len(subset_q)} questions...")

In [ ]:
judge_models = [
    "anthropic.claude-sonnet-4-20250514-v1:0",
    "anthropic.claude-haiku-4-5-20251001-v1:0",
    "anthropic.claude-3-haiku-20240307-v1:0",
]

judge_comparison = compare_judges(
    pipeline=pipeline,
    test_questions=subset_q,
    ground_truths=subset_gt,
    judge_models=judge_models,
)

print("\n" + "="*60)
print("JUDGE SENSITIVITY RESULTS")
print("="*60)
print(judge_comparison.to_string(index=False))

pipeline.cleanup()

In [ ]:
# Visualisation
metrics_names = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
short_names = ['Sonnet 4', 'Haiku 4.5', 'Haiku 3']

x = np.arange(len(metrics_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, (_, row) in enumerate(judge_comparison.iterrows()):
    values = [row[m] for m in metrics_names]
    ax.bar(x + i * width, values, width, label=short_names[i], color=colors[i])

ax.set_ylabel('Score')
ax.set_title('Impact du choix du LLM-juge sur les scores RAGAS\n(même réponses, juges différents)')
ax.set_xticks(x + width)
ax.set_xticklabels([m.replace('_', '\n') for m in metrics_names])
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../data/results/judge_sensitivity.png', dpi=150)
plt.show()

# Calcul de la variance entre juges
for m in metrics_names:
    values = judge_comparison[m].values
    print(f"{m}: range = {values.max() - values.min():.4f} (min={values.min():.4f}, max={values.max():.4f})")

## 2. Comparaison RAGAS vs Évaluation Humaine

On évalue manuellement 10 réponses et on compare avec les scores RAGAS.

**Protocole** : Pour chaque réponse, on note de 0 à 1 :
- Faithfulness : la réponse est-elle factuelle par rapport au contexte ?
- Relevancy : la réponse répond-elle bien à la question ?

In [ ]:
human_scores_path = '../data/testset/human_scores.json'

try:
    with open(human_scores_path) as f:
        human_scores = json.load(f)
    print(f"Loaded {len(human_scores)} human evaluations")
except FileNotFoundError:
    print("human_scores.json not found!")
    print("\nCréez le fichier data/testset/human_scores.json avec vos évaluations manuelles.")
    print("Format : liste de 10 objets avec faithfulness, relevancy, precision, recall (0 à 1)")
    print("\nTemplate :")
    template = [{"faithfulness": 0.0, "relevancy": 0.0, "precision": 0.0, "recall": 0.0} for _ in range(10)]
    print(json.dumps(template[:3], indent=2))
    print("...")

## 3. Identification des cas d'échec de RAGAS

In [ ]:
# Reconstruire le pipeline et analyser question par question
config = EXPERIMENT_CONFIGS[1]
pipeline = RAGPipeline(config)
pipeline.load_documents(documents)
pipeline.chunk_documents()
pipeline.build_index()

# Générer les réponses pour les 10 premières questions
detailed_results = []
for q, gt in zip(subset_q, subset_gt):
    r = pipeline.query(q)
    r['ground_truth'] = gt
    detailed_results.append(r)

# Afficher pour inspection manuelle
for i, r in enumerate(detailed_results):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {r['question']}")
    print(f"\nAnswer: {r['answer'][:200]}...")
    print(f"\nGround Truth: {r['ground_truth'][:200]}...")
    print(f"\nContexts retrieved: {len(r['contexts'])}")
    print(f"Latency: {r['latency']:.2f}s")

pipeline.cleanup()

In [ ]:
print("""
TYPES DE CAS D'ÉCHEC À DOCUMENTER DANS L'ARTICLE :

1. Faux positifs de faithfulness :
   - RAGAS donne un score élevé mais la réponse contient une erreur factuelle
   - Souvent sur les chiffres/dates proches mais incorrects

2. Context recall sous-estimé :
   - Le ground truth est paraphrasé → RAGAS ne reconnaît pas l'équivalence

3. Incohérence entre juges :
   - Un modèle juge "fidèle" ce qu'un autre juge "non-fidèle"
   - Plus fréquent sur les réponses ambiguës/partielles

4. Coût de l'évaluation :
   - RAGAS fait ~4-8 appels LLM par question par métrique
   - Sur 20 questions × 4 métriques = ~200+ appels API
""")